#Evidencia de Aprendizaje 3: Procesamiento Distribuido, Optimizacion y Cierre del caso

**Proyecto:** Wanderbricks
**Estudiante:** Andres Felipe Agudelo Salazar
**Entorno:** Databricks PaaS / Unity Catalog
**Enlace al Repositorio:** 

----


## Seccion 1: Introduccion y Pregunta de Negocio

### Seccion 1.1 Objetivo del Proyecto
El objetivo de esta fase es consolidar la capa **oro** del Lakehouse de Wanderbricks para consumo analitico directo, medir y optimizar tecnicamente una operacion distribuida costosa en Apache Spark, y realizar el cierre del caso analizando arquitecturas de datos empresariales.

### Seccion 1.2 Pregunta de Negocio Seleccionada
¿Cual es el volumen total de reservas, ingresos generados y calificacion promedio agrupados por pais de origen del usuario y estado de la reserva?  

## Seccion 2: Capa Oro y Recorrido del Dato (Pipeline Final)

### 2.1 Construccion de la Tabla Analitica Oro (`oro_consolidado`)
Se consolida la tabla en la capa **Oro** respondiendo a la pregunta de negocio seleccionada. La tabla transforma identificadores tecnicos en atributos legibles para los tomadores de decisiones y calcula metricas clave total de reservas, ingresos y calificacion promedio.     

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.oro_consolidado AS SELECT 
    u.country AS pais_usuario,
    b.status AS estado_reserva,
    COUNT(DISTINCT b.booking_id) AS total_reservas,
    ROUND(SUM(CAST(p.amount AS DOUBLE)), 2) AS total_ingresos,
    ROUND(AVG(CAST(r.rating AS DOUBLE)), 2) AS calificacion_promedio
FROM workspace.mi_lakehouse.bronce_bookings b
JOIN workspace.mi_lakehouse.bronce_users u ON b.user_id = u.user_id
LEFT JOIN workspace.mi_lakehouse.bronce_payments p ON b.booking_id = p.booking_id
LEFT JOIN workspace.mi_lakehouse.bronce_reviews r ON b.booking_id = r.booking_id
GROUP BY u.country, b.status
ORDER BY total_ingresos DESC;





num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM workspace.default.oro_consolidado;

pais_usuario,estado_reserva,total_reservas,total_ingresos,calificacion_promedio
India,pending,5919,3166780.48,2.99
China,pending,5713,3018946.05,3.01
India,confirmed,3333,2785140.71,3.02
China,confirmed,3249,2565697.38,3.04
India,completed,1368,1233666.53,3.01
China,completed,1304,1183390.39,2.96
United States,pending,1409,735156.25,2.99
Indonesia,pending,1158,668165.62,3.0
United States,confirmed,785,629140.24,3.05
Pakistan,pending,995,569822.95,3.01


### 2.2 Trazabilidad y Recorrido del Dato por Capas
Se documenta la cantidad de registros en cada etapa del pipeline para garantizar la calidad y auditoría de la información:

| Capa | Tabla / Entidad | Propósito de la Capa | Justificación de Variación en Conteo |
| :--- | :--- | :--- | :--- |
| **Bronce** | `bronce_bookings` | Ingesta cruda de reservas desde fuentes JSON/CSV. | Almacena el total de registros brutos capturados sin filtrado. |
| **Bronce** | `bronce_users` | Ingesta cruda de usuarios registrados. | Registros completos de cuentas de usuario creadas en la plataforma. |
| **Plata** | `silver_bookings` | Datos limpios, estructurados y validados. | Se eliminan registros duplicados y nulos en identificadores clave (`booking_id`). |
| **Oro** | `oro_consolidado` | Tabla agregada de negocio para reportes. | Se reducen filas al agrupar los datos (*GROUP BY*) por `pais_usuario` y `estado_reserva`. |

## SECCION 3: Medicion Inicial y Plan de Ejecucion (SIN OPTIMIZAR)

A continuacion se define la funcion de medicion de tiempos 

In [0]:
import time
import numpy as np

def medir_tiempo_ejecucion(query_df, nombre_consulta="Consulta"):
    tiempos = []
    print(f"=== Iniciando mediciones para {nombre_consulta} ===") 

    #1. EJECUCION DE CALENTAMIENTO (WARM UP) -SE DESCARTARA
    start = time.time()
    query_df.collect()
    warmup_time = time.time() - start
    print(f"EJECUCION DE CALENTAMIENTO (FUE DESCARTADA): {warmup_time:.3f} s")

    #2. LAS 3 MEDICIONES QUE SON OFICIALES
    for i in range(1, 4):
        start = time.time()
        query_df.collect()
        duracion = time.time() - start
        tiempos.append(duracion)
        print(f"Medicion {i}: {duracion:.3f} s")

    mediana = float(np.median(tiempos))
    print(f" --> Mediana reportafa: {mediana:.3f} s\n")
    return mediana 



In [0]:
#1 LA CONSULTA DISTRIBUIDA DE CRUCE (JOIN) ENTRE RESERVAS Y USUARIOS

query_inicial = spark.sql("""
    SELECT 
        b.booking_id,
        b.status,
        u.name,
        u.country
    FROM workspace.mi_lakehouse.bronce_bookings b
    JOIN workspace.mi_lakehouse.bronce_users u ON b.user_id = u.user_id """)

#2. Plan de ejecucion inicial con .explain()
print(" === PLAN DE EJECUCION INICIAL (ANTES DE SER OPTIMIZADO) ===")
query_inicial.explain(True)

#3. Medicion de tiempos iniciales
mediana_inicial = medir_tiempo_ejecucion(query_inicial, "CONSULTA INICIAL (SIN AVER SIDO OPTIMIZADO)")

 === PLAN DE EJECUCION INICIAL (ANTES DE SER OPTIMIZADO) ===
== Parsed Logical Plan ==
'Project ['b.booking_id, 'b.status, 'u.name, 'u.country]
+- 'Join Inner, ('b.user_id = 'u.user_id)
   :- 'SubqueryAlias b
   :  +- 'UnresolvedRelation [workspace, mi_lakehouse, bronce_bookings], [], false
   +- 'SubqueryAlias u
      +- 'UnresolvedRelation [workspace, mi_lakehouse, bronce_users], [], false

== Analyzed Logical Plan ==
booking_id: bigint, status: string, name: string, country: string
Project [booking_id#12216L, status#12223, name#12228, country#12229]
+- Join Inner, (user_id#12217L = user_id#12226L)
   :- SubqueryAlias b
   :  +- SubqueryAlias workspace.mi_lakehouse.bronce_bookings
   :     +- Relation workspace.mi_lakehouse.bronce_bookings[booking_id#12216L,user_id#12217L,property_id#12218L,check_in#12219,check_out#12220,guests_count#12221,total_amount#12222,status#12223,created_at#12224,updated_at#12225] parquet
   +- SubqueryAlias u
      +- SubqueryAlias workspace.mi_lakehouse.bro

In [0]:
from pyspark.sql.functions import broadcast
import time
import numpy as np

# 1. Aseguramos la función de medición
def medir_tiempo_ejecucion(query_df, nombre_consulta="Consulta"):
    tiempos = []
    print(f"=== Iniciando mediciones para: {nombre_consulta} ===")
    
    # Calentamiento
    start = time.time()
    query_df.collect()
    warmup_time = time.time() - start
    print(f"Ejecución de calentamiento (Descartada): {warmup_time:.3f} s")
    
    # Tres mediciones
    for i in range(1, 4):
        start = time.time()
        query_df.collect()
        duracion = time.time() - start
        tiempos.append(duracion)
        print(f"Medición {i}: {duracion:.3f} s")
    
    mediana = float(np.median(tiempos))
    print(f"--> MEDIANA REPORTADA: {mediana:.3f} s\n")
    return mediana

# 2. Cargar tablas y construir la consulta optimizada con Broadcast
df_bookings = spark.table("workspace.mi_lakehouse.bronce_bookings")
df_users = spark.table("workspace.mi_lakehouse.bronce_users")

query_optimizada = df_bookings.join(
    broadcast(df_users),
    df_bookings.user_id == df_users.user_id,
    "inner"
).select(
    df_bookings.booking_id,
    df_bookings.status,
    df_users.name,
    df_users.country
)

# 3. Mostrar el plan de ejecución optimizado
print("=== PLAN DE EJECUCIÓN OPTIMIZADO (CON BROADCAST JOIN) ===")
query_optimizada.explain(True)

# 4. Medir los tiempos
mediana_optimizada = medir_tiempo_ejecucion(query_optimizada, "Consulta Optimizada (Broadcast Join)")

=== PLAN DE EJECUCIÓN OPTIMIZADO (CON BROADCAST JOIN) ===
== Parsed Logical Plan ==
'Project ['booking_id, 'status, 'name, 'country]
+- 'Join Inner, '`==`('user_id, 'user_id)
   :- 'UnresolvedRelation [workspace, mi_lakehouse, bronce_bookings], [], false
   +- 'UnresolvedHint broadcast
      +- 'UnresolvedRelation [workspace, mi_lakehouse, bronce_users], [], false

== Analyzed Logical Plan ==
booking_id: bigint, status: string, name: string, country: string
Project [booking_id#11589L, status#11596, name#11609, country#11610]
+- Join Inner, (user_id#11590L = user_id#11607L)
   :- SubqueryAlias workspace.mi_lakehouse.bronce_bookings
   :  +- Relation workspace.mi_lakehouse.bronce_bookings[booking_id#11589L,user_id#11590L,property_id#11591L,check_in#11592,check_out#11593,guests_count#11594,total_amount#11595,status#11596,created_at#11597,updated_at#11598] parquet
   +- ResolvedHint (strategy=broadcast)
      +- SubqueryAlias workspace.mi_lakehouse.bronce_users
         +- Relation workspa

%md
### 4.3 Análisis de Resultados y Comparación de Planes

#### ¿Por qué mejoró el rendimiento?
Al utilizar la función `broadcast()`, obligamos a Apache Spark a enviar una copia completa de la tabla pequeña (`bronce_users`) a la memoria de cada uno de los nodos de trabajo (*Workers*). En el plan de ejecución inicial (`.explain()`), Spark utilizaba un operador de cruce por intercambio masivo (*Shuffle Hash Join* o *SortMergeJoin*), el cual requiere mover datos a través de la red para agrupar las filas por `user_id`. Con el operador **BroadcastHashJoin**, se elimina la fase de *Shuffle*, haciendo el cruce directamente en la memoria local de cada ejecutor, lo que reduce drásticamente la latencia de red y el tiempo de CPU.

#### ¿En qué casos esta optimización sería contraproducente?
La técnica de **Broadcast Join** resulta contraproducente o perjudicial bajo dos escenarios principales:
1. **Tamaño excesivo de la tabla difundida:** Si la tabla que se intenta enviar mediante *broadcast* supera la memoria asignada a los controladores (*Driver*) o a los ejecutores (*Workers*) (por defecto > 10 MB o el límite de memoria asignado), provocará un fallo crítico del cluster por **Out of Memory Error (OOM)**.
2. **Tablas de gran volumen que cambian dinámicamente:** Cuando ambas tablas son gigantescas (de cientos de gigabytes o terabytes), intentar transmitir una de ellas ahogará el ancho de banda del cluster y consumirá toda la RAM de los ejecutores, haciendo que el *Shuffle Join* tradicional sea una alternativa mucho más segura y escalable. 

## Seccion 5: Integracion de Paradigmas y Pregunta Orientadora

### 5.1 Pregunta Orientadora del Curso
**¿Como integrar en una unica plataforma corporativa el procesamiento de datos proveniente3s de la web de wanderbricks, interacciones en redes sociales y mensajes ?**

### 5.2 Propuesta de Arquitectura Lakehouse Unificada
Para consolidar estas tres fuentes heterogeneas sin crar silos de informacion, se propone una **Arquitectura Medallon (Lakehouse)** basada en la integracion de patrones de procesamiento Batch y Streaming (Arquitectura Kappa/Delta):

1. **Ingesta de Datos (Capa Bronce):**
   * **Web Wanderbricks (Batch/Micro-batch):** Ingesta de datos estructurados/semi-estructurados (reservas, pagos, usuarios) almacenados en Delta Lake mediante conectores relacionales (JDBC) y archivos JSON/Parquet.
   * **Redes Sociales - Facebook, Instagram, TikTok (Streaming/API):** Ingesta mediante conectores Event-Driven (Apache Kafka / Databricks Auto Loader) que consumen los webhooks de las APIs oficiales (Graph API, TikTok Marketing API). Se almacenan los eventos brutos (*likes*, comentarios, alcance, *clicks*) en formato JSON crudo en la capa Bronce.
   * **WhatsApp Business API (Streaming en Tiempo Real):** Captura de eventos de mensajería (consultas de clientes, confirmaciones, bots) enviados vía Webhooks a un *message broker* (Kafka/Event Hubs) y procesados con *Structured Streaming* directo a la capa Bronce.

   2. **Procesamiento y Limpieza (Capa Plata):**
   * Se aplican transformaciones en PySpark para desempaquetar las estructuras JSON entrantes, eliminar duplicados, aplicar esquemas estrictos y anonimizar datos personales (cumplimiento GDPR/Habeas Data).

   * Se realiza la **unificacion de identidades** (Customer 360) cruzando el `user_id` o numero telefonico/email para vincular la actividad en redes sociales y Whatsapp con el historial de reservas en la web.

   3. **Consolidacion y Analitica Avanzada (Capa Oro): **
   * **Modelado Dimensional:** Se construyen tablas agregadas Oro optimizadas para consumo de herramientas de BI (Power BI, Databricks SQL Dashboards) y modelos de Machine Learning (Analisis de Sentimiento en mensajes de Wahtsapp y comentarios de redes sociales, atribucion de campañas publicitarias ).



## Seccion 6:Conclusiones y Declaracion de Integrante

### 6.1 Conclusiones del Proyecto
1. **Consolidacion Analitica:** La implementacion de la capa **Oro** (`oro_consolidado`) permitio abstraer la complejidad tecnica de las capas crudas y entregar una vista agregada lista para responder preguntas criticas de negocio sobre ingresos y calificaciones por pais. 
2. **Impacto de la Optimizacion:** Mediante la captura del plan de ejecucion (`.explain()`) y la aplicacion de la tecnica **Broadcast Join**, se demostro como reducir el intercambio masivo de datos por red (*SHUFFLE*), optimizando el tiempo de ejecucion en consultas distributed joins con tablas de dimension pequeñas.
3. **Escalabilidad Lakehouse:** La arquitectura Medallon demostro ser el patron idoneo para escalar desde ingestas estructuradas hasta modelos analíticos empresariales con gobernanza centralizada vía Unity Catalog.

---


### 6.2 Declaracion de integrantes y Transparencia en IA
* **Estudiante:** Andres Felipe Agudelo Salazar
* **Porcentaje de Participacion:** 100%
* **Declaracion del uso de Inteligencia Artificial:** Se utilizo asistencia de modelos de lenguaje (IA GENERATIVA) como apoyo tecnico en la estructuracion de la sintaxis PySpark/SQL, redaccion de documentacion analitica y optimizacion de consultas, realizando la validacion, ejecucion y verificacion experimental directa en el entorno PaaS de Databricks .

